In [3]:
from pathlib import Path

import pandas as pd
import numpy as np

# Project root is the parent of the notebooks folder
PROJECT_ROOT = Path.cwd().parent

RAW_FILE = PROJECT_ROOT / "data" / "raw" / "superstore.csv"

# The Superstore CSV is Windows-1252 encoded, not UTF-8
df = pd.read_csv(RAW_FILE, encoding="cp1252")

print("Dataset loaded successfully!")

Dataset loaded successfully!


In [4]:
df.shape

(9994, 21)

In [5]:
df.columns.tolist()

['Row ID',
 'Order ID',
 'Order Date',
 'Ship Date',
 'Ship Mode',
 'Customer ID',
 'Customer Name',
 'Segment',
 'Country',
 'City',
 'State',
 'Postal Code',
 'Region',
 'Product ID',
 'Category',
 'Sub-Category',
 'Product Name',
 'Sales',
 'Quantity',
 'Discount',
 'Profit']

In [6]:
df.dtypes

Row ID             int64
Order ID             str
Order Date           str
Ship Date            str
Ship Mode            str
Customer ID          str
Customer Name        str
Segment              str
Country              str
City                 str
State                str
Postal Code        int64
Region               str
Product ID           str
Category             str
Sub-Category         str
Product Name         str
Sales            float64
Quantity           int64
Discount         float64
Profit           float64
dtype: object

In [7]:
df.isnull().sum()

Row ID           0
Order ID         0
Order Date       0
Ship Date        0
Ship Mode        0
Customer ID      0
Customer Name    0
Segment          0
Country          0
City             0
State            0
Postal Code      0
Region           0
Product ID       0
Category         0
Sub-Category     0
Product Name     0
Sales            0
Quantity         0
Discount         0
Profit           0
dtype: int64

In [8]:
df.duplicated().sum()

np.int64(0)

In [9]:
df["Customer ID"].nunique()

793

In [10]:
df["Product ID"].nunique()


1862

In [11]:
df["Product Name"].nunique()

1850

In [12]:
df["Segment"].unique()

<StringArray>
['Consumer', 'Corporate', 'Home Office']
Length: 3, dtype: str

In [13]:
df["Region"].unique()

<StringArray>
['South', 'West', 'Central', 'East']
Length: 4, dtype: str

In [15]:
df["Category"].unique()

<StringArray>
['Furniture', 'Office Supplies', 'Technology']
Length: 3, dtype: str

In [16]:
df["Sub-Category"].unique()

<StringArray>
[  'Bookcases',      'Chairs',      'Labels',      'Tables',     'Storage',
 'Furnishings',         'Art',      'Phones',     'Binders',  'Appliances',
       'Paper', 'Accessories',   'Envelopes',   'Fasteners',    'Supplies',
    'Machines',     'Copiers']
Length: 17, dtype: str

In [17]:
df["Ship Mode"].unique()

<StringArray>
['Second Class', 'Standard Class', 'First Class', 'Same Day']
Length: 4, dtype: str

In [18]:
df["Order Date"].min()

'1/1/2017'

In [19]:
df["Order Date"].max()

'9/9/2017'

In [20]:
df["Order Date"] = pd.to_datetime(df["Order Date"])

In [21]:
df["Year"] = df["Order Date"].dt.year
df["Month"] = df["Order Date"].dt.month
df["Month_Name"] = df["Order Date"].dt.month_name()
df["Quarter"] = df["Order Date"].dt.quarter

In [22]:
df["Revenue"] = df["Sales"]

In [23]:
df["COGS"] = df["Sales"] - df["Profit"]

In [24]:
df["Profit_Margin"] = (
    df["Profit"] / df["Sales"]
) * 100

In [25]:
df["Profit_Margin"] = np.where(
    df["Sales"] != 0,
    (df["Profit"] / df["Sales"]) * 100,
    0
)

# 2. Data Cleaning & Validation

In [26]:
df_clean = df.copy()

print("Original shape:", df.shape)
print("Cleaning shape:", df_clean.shape)

Original shape: (9994, 28)
Cleaning shape: (9994, 28)


In [ ]:
df_clean.columns = (
    df_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

df_clean.columns.tolist()

In [28]:
df_clean["order_date"] = pd.to_datetime(
    df_clean["order_date"],
    errors="coerce"
)

df_clean["ship_date"] = pd.to_datetime(
    df_clean["ship_date"],
    errors="coerce"
)

In [29]:
print(df_clean["order_date"].dtype)
print(df_clean["ship_date"].dtype)

datetime64[us]
datetime64[us]


In [30]:
print("Earliest order:", df_clean["order_date"].min())
print("Latest order:", df_clean["order_date"].max())

print("Earliest shipment:", df_clean["ship_date"].min())
print("Latest shipment:", df_clean["ship_date"].max())

Earliest order: 2014-01-03 00:00:00
Latest order: 2017-12-30 00:00:00
Earliest shipment: 2014-01-07 00:00:00
Latest shipment: 2018-01-05 00:00:00


In [31]:
invalid_shipments = df_clean[
    df_clean["ship_date"] < df_clean["order_date"]
]

print("Shipments before order:", len(invalid_shipments))

Shipments before order: 0


In [32]:
df_clean[
    ["sales", "quantity", "discount", "profit"]
].dtypes

sales       float64
quantity      int64
discount    float64
profit      float64
dtype: object

In [33]:
numeric_columns = [
    "sales",
    "quantity",
    "discount",
    "profit"
]

for col in numeric_columns:
    df_clean[col] = pd.to_numeric(
        df_clean[col],
        errors="coerce"
    )

In [34]:
df_clean[numeric_columns].isna().sum()

sales       0
quantity    0
discount    0
profit      0
dtype: int64

In [35]:
print("Quantity <= 0:",
      (df_clean["quantity"] <= 0).sum())

Quantity <= 0: 0


In [36]:
print("Sales < 0:",
      (df_clean["sales"] < 0).sum())

Sales < 0: 0


In [37]:
print("Discount < 0:",
      (df_clean["discount"] < 0).sum())

print("Discount > 100%:",
      (df_clean["discount"] > 1).sum())

Discount < 0: 0
Discount > 100%: 0


In [38]:
print("Negative profit:",
      (df_clean["profit"] < 0).sum())

Negative profit: 1871


In [42]:
# Make sure hyphenated names (e.g. "sub-category") use underscores
df_clean.columns = df_clean.columns.str.replace("-", "_")

text_columns = [
    "order_id",
    "ship_mode",
    "customer_id",
    "customer_name",
    "segment",
    "country",
    "city",
    "state",
    "region",
    "product_id",
    "category",
    "sub_category",
    "product_name"
]

for col in text_columns:
    df_clean[col] = df_clean[col].str.strip()